In [34]:
import pandas as pd
import numpy as np
from datetime import time

start_date = "2005-03-01"
end_date = "2006-03-01"
df = pd.read_csv('data/SPY_15min_2002-01_to_2006-08.csv')
df2 = pd.read_csv('data/SPY_15min_2020-01_to_2022-01.csv')


#df['datetime'] = df['datetime'].dt.tz_localize('US/Eastern')
df['datetime'] = pd.to_datetime(df['Unnamed: 0'], utc=True)
print(df['Unnamed: 0'].head())
print(df['Unnamed: 0'].dtype)


df.set_index('datetime', inplace=True)
df.index = pd.DatetimeIndex(df.index)  # This ensures time-aware index
df = df.asfreq('15min')  # Ensure the index is at 15-minute frequency
df.index = df.index.tz_convert("Europe/Berlin")  # Convert to CET timezone
df = df.between_time("15:30", "22:00")
df = df[df.index.weekday < 5] # Keep only Monday (0) through Friday (4)

df["y^2"] = (df["close"].apply(lambda x: np.log(x)).diff()) ** 2
df = df.loc[pd.Timestamp(start).tz_localize("Europe/Berlin"):pd.Timestamp(end_date).tz_localize("Europe/Berlin")]

df.drop(columns=['Unnamed: 0'], inplace=True)
df.interpolate(method='time', inplace=True)


df["day_of_week"] = df.index.dayofweek
df["hour_of_day"] = df.index.hour

day_dummies = pd.get_dummies(df["day_of_week"], prefix="day")
hour_dummies = pd.get_dummies(df["hour_of_day"], prefix="hour")



df = pd.concat([df, day_dummies, hour_dummies], axis=1)
df

#todo filter for correct dates with dummy variables

0    2002-01-02 09:30:00-05:00
1    2002-01-02 09:45:00-05:00
2    2002-01-02 10:00:00-05:00
3    2002-01-02 10:15:00-05:00
4    2002-01-02 10:30:00-05:00
Name: Unnamed: 0, dtype: object
object


,open,high,low,close,volume,y^2,day_of_week,hour_of_day,day_0,day_1,...,day_3,day_4,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22
datetime,,,,,,,,,,,,,,,,,,,,,
2005-03-01 15:30:00+01:00,83.1154,83.4113,83.1154,83.3769,3869200.0,1.779731e-05,1,15,False,True,...,False,False,True,False,False,False,False,False,False,False
2005-03-01 15:45:00+01:00,83.3769,83.4044,83.2805,83.3494,2046100.0,1.088221e-07,1,15,False,True,...,False,False,True,False,False,False,False,False,False,False
2005-03-01 16:00:00+01:00,83.3494,83.5971,83.2874,83.4870,5279800.0,2.720918e-06,1,16,False,True,...,False,False,False,True,False,False,False,False,False,False
2005-03-01 16:15:00+01:00,83.4870,83.5145,83.4044,83.4113,2816800.0,8.229017e-07,1,16,False,True,...,False,False,False,True,False,False,False,False,False,False
2005-03-01 16:30:00+01:00,83.4113,83.5489,83.3906,83.4870,1562300.0,8.229017e-07,1,16,False,True,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006-02-28 21:00:00+01:00,89.8593,89.9154,89.7263,89.7543,3240400.0,1.190879e-06,1,21,False,True,...,False,False,False,False,False,False,False,False,True,False
2006-02-28 21:15:00+01:00,89.7543,89.9154,89.7473,89.9014,1836600.0,2.681659e-06,1,21,False,True,...,False,False,False,False,False,False,False,False,True,False
2006-02-28 21:30:00+01:00,89.9014,89.9854,89.8453,89.9644,1822900.0,4.907315e-07,1,21,False,True,...,False,False,False,False,False,False,False,False,True,False


In [52]:

df_news = pd.read_csv('data/WhatMovesMarkets_eventdatabase.csv')
df_news["eventstart_CET"] = pd.to_datetime(df_news["eventstart_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
df_news["eventend_CET"] = pd.to_datetime(df_news["eventend_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
df_news.set_index('eventstart_CET', inplace=True)
df_news.drop(columns=["eventstart", "eventend"], inplace=True)
df_news.index = df_news.index.floor("15min")  # Round down to the nearest 15 minutes

# Correct time window creation according to paper
time_mask = (time(15, 30) <= df_news.index.time) & (df_news.index.time <= time(22, 0))# Helper

# Case 1: Events with an end time and time_mask = True
mask1 = df_news["eventend_CET"].notna() & time_mask
df_news.loc[mask1, "window_start"] = df_news.index[mask1] - pd.Timedelta(minutes=20)
df_news.loc[mask1, "window_end"]   = df_news.index[mask1] + pd.Timedelta(minutes=20)

# Case 2: Events with an end time and time_mask = False
mask2 = df_news["eventend_CET"].notna() & ~time_mask
df_news.loc[mask2, "window_start"] = pd.to_datetime(df_news.index.date[mask2].astype(str) + " 21:40").tz_localize("Etc/GMT-1")
df_news.loc[mask2, "window_end"]   = pd.to_datetime(df_news.index.date[mask2].astype(str) + " 15:50").tz_localize("Etc/GMT-1")

# Case 3: Events without end time and time_mask = True
mask3 = df_news["eventend_CET"].isna() & time_mask
df_news.loc[mask3, "window_start"] = df_news.index[mask3] - pd.Timedelta(minutes=15)
df_news.loc[mask3, "window_end"]   = df_news.index[mask3] + pd.Timedelta(minutes=30)

# Case 4: Events without end time and time_mask = False
mask4 = df_news["eventend_CET"].isna() & ~time_mask
df_news.loc[mask4, "window_start"] = pd.to_datetime(df_news.index.date[mask4].astype(str) + " 21:45").tz_localize("Etc/GMT-1")
df_news.loc[mask4, "window_end"]   = pd.to_datetime(df_news.index.date[mask4].astype(str) + " 16:00").tz_localize("Etc/GMT-1")
df_news

,name,type,subtype,subsubtype,description,source,scheduled,eventend_CET,window_start,window_end
eventstart_CET,,,,,,,,,,
2002-03-01 07:00:00+01:00,FI Consumer Confidence,Macro Release,FI,Consumer Confidence,NaN,Bloomberg,1,NaT,2002-03-01 21:45:00+01:00,2002-03-01 16:00:00+01:00
2002-03-01 07:30:00+01:00,CH CPI,Macro Release,CH,CPI,NaN,Bloomberg,1,NaT,2002-03-01 21:45:00+01:00,2002-03-01 16:00:00+01:00
2002-03-01 09:00:00+01:00,IT CPI,Macro Release,IT,CPI,NaN,Bloomberg,1,NaT,2002-03-01 21:45:00+01:00,2002-03-01 16:00:00+01:00
2002-03-01 10:30:00+01:00,UK Monetary Aggregates,Macro Release,UK,Monetary Aggregates,NaN,Bloomberg,1,NaT,2002-03-01 21:45:00+01:00,2002-03-01 16:00:00+01:00
2002-03-01 12:00:00+01:00,EA Retail Sales & EA Retail Trade,Macro Release,EA,Retail Sales & EA Retail Trade,NaN,Bloomberg,1,NaT,2002-03-01 21:45:00+01:00,2002-03-01 16:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...
2020-08-31 17:30:00+01:00,US Auction Result Bill,Auction,US Result,Bill,CUSIP 9127964F3,https://www.treasurydirect.gov/instit/annceres...,1,NaT,2020-08-31 17:15:00+01:00,2020-08-31 18:00:00+01:00
2020-08-31 17:30:00+01:00,US Auction Result Bill,Auction,US Result,Bill,CUSIP 912796TU3,https://www.treasurydirect.gov/instit/annceres...,1,NaT,2020-08-31 17:15:00+01:00,2020-08-31 18:00:00+01:00
2020-09-01 02:00:00+01:00,IE Investec Manufacturing PMI,Macro Release,IE,Investec Manufacturing PMI,NaN,Bloomberg,1,NaT,2020-09-01 21:45:00+01:00,2020-09-01 16:00:00+01:00


In [54]:
#todo make this faster
print(f"creating {len(df_news["subsubtype"].unique())} event dummie variables")
new_cols = {}
for subsubtype, df_type in df_news.groupby("subsubtype"):
    mask = pd.Series(False, index=df.index)
    for _, row in df_type.iterrows():
        mask |= (df.index >= row["window_start"]) & (df.index < row["window_end"])
    new_cols[f"D_{subsubtype}"] = mask

df = pd.concat([df, pd.DataFrame(new_cols)])
df

creating 128 event dummie variables


,open,high,low,close,volume,y^2,day_of_week,hour_of_day,day_0,day_1,...,D_Trade & Current Account Balance,D_Trade Balance,D_Trade Balance Non-Eu (Euros),D_Trade Data (CNY),D_Ulster Bank Construction PMI,D_Unemployment Net,D_Unemployment Rate,D_University of Michigan Surveys,D_Weekly Financial Statement,D_Wholesale Price Index
datetime,,,,,,,,,,,,,,,,,,,,,
2005-03-01 15:30:00+01:00,83.1154,83.4113,83.1154,83.3769,3869200.0,1.779731e-05,1.0,15.0,False,True,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 15:45:00+01:00,83.3769,83.4044,83.2805,83.3494,2046100.0,1.088221e-07,1.0,15.0,False,True,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:00:00+01:00,83.3494,83.5971,83.2874,83.4870,5279800.0,2.720918e-06,1.0,16.0,False,True,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:15:00+01:00,83.4870,83.5145,83.4044,83.4113,2816800.0,8.229017e-07,1.0,16.0,False,True,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:30:00+01:00,83.4113,83.5489,83.3906,83.4870,1562300.0,8.229017e-07,1.0,16.0,False,True,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006-02-28 21:00:00+01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
2006-02-28 21:15:00+01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
2006-02-28 21:30:00+01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
